# Pillar 2 — SQL & Data Visualization Validation

This notebook validates the implementation for Pillar 2 of the StackUp Engineering Academy Data Engineering Assessment.

The validation is performed chronologically across:

- Task 2.1 — SQL business questions
- Task 2.2 — Transaction ETL pipeline
- Task 2.3 — Query optimisation
- Task 2.4 — Dashboard data validation

The notebook is used as a validation and evidence workspace. Final SQL, ETL code,
and dashboard deliverables are maintained in their respective submission files.

## 1. Environment Setup

Import the libraries required for data loading, SQL validation, and analysis.

In [20]:
import pandas as pd
import json
import time
from pathlib import Path
import duckdb

In [21]:
# Repository paths

REPO_ROOT = Path.cwd().parent

DATA_DIR = REPO_ROOT / "datasets"

FOUNDATIONS_OUTPUT_DIR = (
    REPO_ROOT
    / "outputs"
    / "results"
    / "vrinda-daga"
    / "01_foundations"
)

TASK2_OUTPUT_DIR = (
    REPO_ROOT
    / "outputs"
    / "results"
    / "vrinda-daga"
    / "02_sql_and_viz"
)

TASK2_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Repository root:", REPO_ROOT)
print("Datasets:", DATA_DIR)
print("Foundations outputs:", FOUNDATIONS_OUTPUT_DIR)
print("Task 2 outputs:", TASK2_OUTPUT_DIR)

Repository root: c:\Users\vrinda.daga\projects\stackup-engineering-academy_assessment
Datasets: c:\Users\vrinda.daga\projects\stackup-engineering-academy_assessment\datasets
Foundations outputs: c:\Users\vrinda.daga\projects\stackup-engineering-academy_assessment\outputs\results\vrinda-daga\01_foundations
Task 2 outputs: c:\Users\vrinda.daga\projects\stackup-engineering-academy_assessment\outputs\results\vrinda-daga\02_sql_and_viz


## 2. Load Task 1 cleaned datasets

In [22]:
projects_clean_path = (
    FOUNDATIONS_OUTPUT_DIR / "projects_clean.csv"
)

employees_clean_path = (
    FOUNDATIONS_OUTPUT_DIR / "employees_clean.csv"
)

projects_clean = pd.read_csv(projects_clean_path)
employees_clean = pd.read_csv(employees_clean_path)

print("Projects rows:", len(projects_clean))
print("Projects columns:", len(projects_clean.columns))

print("Employees rows:", len(employees_clean))
print("Employees columns:", len(employees_clean.columns))

Projects rows: 500
Projects columns: 17
Employees rows: 1000
Employees columns: 12


## 3. Load raw transaction data

In [23]:
transactions_path = DATA_DIR / "transactions.json"

with open(transactions_path, "r", encoding="utf-8") as f:
    raw_transactions = json.load(f)

print("Raw transaction records:", len(raw_transactions))

print("\nFirst transaction:")
print(raw_transactions[0])

Raw transaction records: 50000

First transaction:
{'transaction_id': 'TXN000001', 'project_id': 'PRJ0053', 'vendor_id': 'VND061', 'vendor_name': 'FleetTrack Pro', 'category': 'Research & Development', 'amount': 86860, 'currency': 'AED', 'transaction_date': '2022-11-16', 'approved_by': 'EMP0169', 'payment_status': 'Paid', 'invoice_ref': 'INV-2022-0001', 'notes': 'Hardware procurement'}


In [24]:
transactions = pd.DataFrame(raw_transactions)

print("Transaction rows:", len(transactions))
print("Transaction columns:", len(transactions.columns))

print("\nTransaction columns:")
print(transactions.columns.tolist())

Transaction rows: 50000
Transaction columns: 12

Transaction columns:
['transaction_id', 'project_id', 'vendor_id', 'vendor_name', 'category', 'amount', 'currency', 'transaction_date', 'approved_by', 'payment_status', 'invoice_ref', 'notes']


## 4. DuckDB connection

In [25]:
con = duckdb.connect()

print("DuckDB connection created successfully.")

DuckDB connection created successfully.


In [26]:
con.register("projects_df", projects_clean)
con.register("employees_df", employees_clean)
con.register("transactions_df", transactions)

print(
    "Projects:",
    con.execute("SELECT COUNT(*) FROM projects_df").fetchone()[0]
)

print(
    "Employees:",
    con.execute("SELECT COUNT(*) FROM employees_df").fetchone()[0]
)

print(
    "Transactions:",
    con.execute("SELECT COUNT(*) FROM transactions_df").fetchone()[0]
)

Projects: 500
Employees: 1000
Transactions: 50000


## 2.1 — SQL Business Questions

### 2.1.1 — Recreate Task 1.2 Star Schema

The Task 1.2 star schema is recreated in the current DuckDB session so that all Task 2 SQL queries can be validated against the same warehouse model.

Tables:
- dim_date
- dim_project
- dim_employee
- dim_vendor
- bridge_employee_project
- fact_transactions

In [27]:
# Reset Task 1.2 tables so the validation notebook can be rerun safely.

con.execute("""
DROP TABLE IF EXISTS fact_transactions;
DROP TABLE IF EXISTS bridge_employee_project;
DROP TABLE IF EXISTS dim_vendor;
DROP TABLE IF EXISTS dim_employee;
DROP TABLE IF EXISTS dim_project;
DROP TABLE IF EXISTS dim_date;
""")

# Dimension: Date
con.execute("""
CREATE TABLE dim_date (
    date_key INTEGER PRIMARY KEY,
    full_date DATE NOT NULL,
    year INTEGER NOT NULL,
    quarter INTEGER NOT NULL,
    month INTEGER NOT NULL,
    month_name VARCHAR(20) NOT NULL,
    week INTEGER NOT NULL,
    day INTEGER NOT NULL,
    day_of_week INTEGER NOT NULL,
    is_weekend BOOLEAN NOT NULL
)
""")

# Dimension: Project
con.execute("""
CREATE TABLE dim_project (
    project_key INTEGER PRIMARY KEY,
    project_id VARCHAR(50) NOT NULL UNIQUE,
    project_name VARCHAR(255),
    department VARCHAR(100),
    status VARCHAR(50),
    start_date DATE,
    end_date DATE,
    budget DECIMAL(15,2),
    actual_cost DECIMAL(15,2),
    project_manager_id VARCHAR(50),
    priority VARCHAR(50),
    region VARCHAR(100),
    budget_variance DECIMAL(15,2),
    is_over_budget BOOLEAN,
    duration_days INTEGER,
    budget_utilisation_pct DECIMAL(10,2),
    status_category VARCHAR(50),
    risk_level VARCHAR(50)
)
""")

# Dimension: Employee — SCD Type 2
con.execute("""
CREATE TABLE dim_employee (
    employee_key INTEGER PRIMARY KEY,
    employee_id VARCHAR(50) NOT NULL,
    full_name VARCHAR(255),
    email VARCHAR(255),
    department VARCHAR(100),
    role VARCHAR(100),
    level VARCHAR(50),
    hire_date DATE,
    salary DECIMAL(15,2),
    manager_id VARCHAR(50),
    region VARCHAR(100),
    status VARCHAR(50),
    years_experience INTEGER,
    valid_from DATE NOT NULL,
    valid_to DATE NOT NULL,
    is_current BOOLEAN NOT NULL,
    change_reason VARCHAR(255)
)
""")

# Dimension: Vendor
con.execute("""
CREATE TABLE dim_vendor (
    vendor_key INTEGER PRIMARY KEY,
    vendor_id VARCHAR(100) NOT NULL UNIQUE,
    vendor_name VARCHAR(255)
)
""")

# Bridge: Employee ↔ Project
con.execute("""
CREATE TABLE bridge_employee_project (
    employee_key INTEGER NOT NULL,
    project_key INTEGER NOT NULL,

    PRIMARY KEY (employee_key, project_key),

    FOREIGN KEY (employee_key)
        REFERENCES dim_employee(employee_key),

    FOREIGN KEY (project_key)
        REFERENCES dim_project(project_key)
)
""")

# Fact: Transactions
con.execute("""
CREATE TABLE fact_transactions (
    transaction_key INTEGER PRIMARY KEY,
    transaction_id VARCHAR(100) NOT NULL UNIQUE,
    project_key INTEGER NOT NULL,
    employee_key INTEGER,
    vendor_key INTEGER,
    date_key INTEGER NOT NULL,
    amount DECIMAL(15,2),
    category VARCHAR(100),
    payment_status VARCHAR(50),

    FOREIGN KEY (project_key)
        REFERENCES dim_project(project_key),

    FOREIGN KEY (employee_key)
        REFERENCES dim_employee(employee_key),

    FOREIGN KEY (vendor_key)
        REFERENCES dim_vendor(vendor_key),

    FOREIGN KEY (date_key)
        REFERENCES dim_date(date_key)
)
""")

print("Six Task 1.2 warehouse tables created successfully.")

Six Task 1.2 warehouse tables created successfully.


In [28]:
# Load staging tables
# Raw employee data is used for SCD2 construction.

con.execute("""
CREATE OR REPLACE TABLE stg_employees AS
SELECT *
FROM read_csv_auto(?)
""", [str(DATA_DIR / "employees.csv")])

# Salary history is required to construct historical employee versions.
con.execute("""
CREATE OR REPLACE TABLE stg_salary_history AS
SELECT *
FROM read_csv_auto(?)
""", [str(DATA_DIR / "employees_salary_history.csv")])

print(
    "stg_employees rows:",
    con.execute("SELECT COUNT(*) FROM stg_employees").fetchone()[0]
)

print(
    "stg_salary_history rows:",
    con.execute("SELECT COUNT(*) FROM stg_salary_history").fetchone()[0]
)

stg_employees rows: 1000
stg_salary_history rows: 1826


In [29]:
# Populate dim_employee — SCD Type 2


con.execute("""
INSERT INTO dim_employee (
    employee_key,
    employee_id,
    full_name,
    email,
    department,
    role,
    level,
    hire_date,
    salary,
    manager_id,
    region,
    status,
    years_experience,
    valid_from,
    valid_to,
    is_current,
    change_reason
)

WITH clean_history AS (
    SELECT *
    FROM (
        SELECT
            h.*,
            ROW_NUMBER() OVER (
                PARTITION BY employee_id, effective_date
                ORDER BY new_salary DESC
            ) AS rn
        FROM stg_salary_history h
    ) ranked_history
    WHERE rn = 1
),

history_with_dates AS (
    SELECT
        h.*,
        LEAD(effective_date) OVER (
            PARTITION BY employee_id
            ORDER BY effective_date
        ) AS next_effective_date
    FROM clean_history h
)

SELECT
    ROW_NUMBER() OVER (
        ORDER BY h.employee_id, h.effective_date
    ) AS employee_key,

    e.employee_id,
    e.full_name,
    e.email,
    e.department,
    h.new_role AS role,
    h.new_level AS level,
    TRY_CAST(e.hire_date AS DATE) AS hire_date,
    h.new_salary AS salary,
    e.manager_id,
    e.region,
    e.status,
    e.years_experience,
    CAST(h.effective_date AS DATE) AS valid_from,

    COALESCE(
        CAST(h.next_effective_date AS DATE),
        DATE '9999-12-31'
    ) AS valid_to,

    CASE
        WHEN h.next_effective_date IS NULL THEN TRUE
        ELSE FALSE
    END AS is_current,

    h.change_reason

FROM history_with_dates h
JOIN stg_employees e
    ON e.employee_id = h.employee_id
""")

# Add employees without salary history
con.execute("""
INSERT INTO dim_employee (
    employee_key,
    employee_id,
    full_name,
    email,
    department,
    role,
    level,
    hire_date,
    salary,
    manager_id,
    region,
    status,
    years_experience,
    valid_from,
    valid_to,
    is_current,
    change_reason
)

SELECT
    (
        SELECT COALESCE(MAX(employee_key), 0)
        FROM dim_employee
    )
    + ROW_NUMBER() OVER (ORDER BY e.employee_id),

    e.employee_id,
    e.full_name,
    e.email,
    e.department,
    e.role,
    e.level,
    TRY_CAST(e.hire_date AS DATE),
    e.salary,
    e.manager_id,
    e.region,
    e.status,
    e.years_experience,

    COALESCE(
        TRY_CAST(e.hire_date AS DATE),
        DATE '1900-01-01'
    ),

    DATE '9999-12-31',
    TRUE,
    NULL

FROM stg_employees e

WHERE NOT EXISTS (
    SELECT 1
    FROM stg_salary_history h
    WHERE h.employee_id = e.employee_id
)
""")

print(
    con.execute("""
        SELECT
            COUNT(*) AS total_records,
            COUNT(DISTINCT employee_id) AS employees,
            SUM(CASE WHEN is_current THEN 1 ELSE 0 END) AS current_records
        FROM dim_employee
    """).df()
)

   total_records  employees  current_records
0           2231       1000           1000.0


In [30]:
# 7. Populate dim_date


con.execute("""
INSERT INTO dim_date (
    date_key,
    full_date,
    year,
    quarter,
    month,
    month_name,
    week,
    day,
    day_of_week,
    is_weekend
)

SELECT
    CAST(STRFTIME(d, '%Y%m%d') AS INTEGER) AS date_key,
    d AS full_date,
    EXTRACT(YEAR FROM d) AS year,
    EXTRACT(QUARTER FROM d) AS quarter,
    EXTRACT(MONTH FROM d) AS month,
    STRFTIME(d, '%B') AS month_name,
    EXTRACT(WEEK FROM d) AS week,
    EXTRACT(DAY FROM d) AS day,
    EXTRACT(DAYOFWEEK FROM d) AS day_of_week,

    CASE
        WHEN EXTRACT(DAYOFWEEK FROM d) IN (0, 6)
        THEN TRUE
        ELSE FALSE
    END AS is_weekend

FROM generate_series(
    DATE '2020-01-01',
    DATE '2030-12-31',
    INTERVAL '1 day'
) AS dates(d)
""")

print(
    con.execute("""
        SELECT
            COUNT(*) AS rows,
            MIN(full_date) AS min_date,
            MAX(full_date) AS max_date
        FROM dim_date
    """).df()
)

   rows   min_date   max_date
0  4018 2020-01-01 2030-12-31


In [31]:
# 8. Populate dim_project


con.execute("""
INSERT INTO dim_project (
    project_key,
    project_id,
    project_name,
    department,
    status,
    start_date,
    end_date,
    budget,
    actual_cost,
    project_manager_id,
    priority,
    region,
    budget_variance,
    is_over_budget,
    duration_days,
    budget_utilisation_pct,
    status_category,
    risk_level
)

SELECT
    ROW_NUMBER() OVER (ORDER BY project_id) AS project_key,
    project_id,
    project_name,
    department,
    status,
    TRY_CAST(start_date AS DATE),
    TRY_CAST(end_date AS DATE),
    budget,
    actual_cost,
    project_manager_id,
    priority,
    region,
    budget_variance,
    is_over_budget,
    duration_days,
    budget_utilisation_pct,
    status_category,
    risk_level

FROM read_csv_auto(?)
""", [str(FOUNDATIONS_OUTPUT_DIR / "projects_clean.csv")])

print(
    con.execute("""
        SELECT COUNT(*) AS project_count
        FROM dim_project
    """).df()
)

   project_count
0            500


In [32]:
# 9. Populate dim_vendor


con.execute("""
INSERT INTO dim_vendor (
    vendor_key,
    vendor_id,
    vendor_name
)

SELECT
    ROW_NUMBER() OVER (ORDER BY vendor_id) AS vendor_key,
    vendor_id,
    MAX(vendor_name) AS vendor_name

FROM read_json_auto(?)

WHERE vendor_id IS NOT NULL

GROUP BY vendor_id
""", [str(DATA_DIR / "transactions.json")])

print(
    con.execute("""
        SELECT COUNT(*) AS vendor_count
        FROM dim_vendor
    """).df()
)

   vendor_count
0            25


In [33]:

# 10. Populate bridge_employee_project


con.execute("""
INSERT INTO bridge_employee_project (
    employee_key,
    project_key
)

SELECT DISTINCT
    e.employee_key,
    p.project_key

FROM read_json_auto(?) t

JOIN dim_employee e
    ON t.approved_by = e.employee_id
    AND e.is_current = TRUE

JOIN dim_project p
    ON t.project_id = p.project_id

WHERE t.approved_by IS NOT NULL
  AND t.project_id IS NOT NULL
""", [str(DATA_DIR / "transactions.json")])

print(
    con.execute("""
        SELECT COUNT(*) AS bridge_rows
        FROM bridge_employee_project
    """).df()
)

   bridge_rows
0        42095


In [34]:
# ============================================================
# 11. Populate fact_transactions
# ============================================================

con.execute("""
INSERT INTO fact_transactions (
    transaction_key,
    transaction_id,
    project_key,
    employee_key,
    vendor_key,
    date_key,
    amount,
    category,
    payment_status
)

SELECT
    ROW_NUMBER() OVER (
        ORDER BY t.transaction_id
    ) AS transaction_key,

    t.transaction_id,

    p.project_key,

    e.employee_key,

    v.vendor_key,

    d.date_key,

    TRY_CAST(t.amount AS DECIMAL(15,2)),
    t.category,
    t.payment_status

FROM read_json_auto(?) t

JOIN dim_project p
    ON t.project_id = p.project_id

LEFT JOIN dim_employee e
    ON t.approved_by = e.employee_id
    AND e.is_current = TRUE

JOIN dim_vendor v
    ON t.vendor_id = v.vendor_id

JOIN dim_date d
    ON TRY_CAST(t.transaction_date AS DATE) = d.full_date
""", [str(DATA_DIR / "transactions.json")])

print(
    con.execute("""
        SELECT
            COUNT(*) AS transaction_rows,
            COUNT(DISTINCT transaction_id) AS unique_transactions
        FROM fact_transactions
    """).df()
)

   transaction_rows  unique_transactions
0             50000                50000


In [35]:
# ============================================================
# 12. Final warehouse validation
# ============================================================

warehouse_counts = con.execute("""
SELECT 'dim_date' AS table_name, COUNT(*) AS row_count
FROM dim_date

UNION ALL

SELECT 'dim_project', COUNT(*)
FROM dim_project

UNION ALL

SELECT 'dim_employee', COUNT(*)
FROM dim_employee

UNION ALL

SELECT 'dim_vendor', COUNT(*)
FROM dim_vendor

UNION ALL

SELECT 'bridge_employee_project', COUNT(*)
FROM bridge_employee_project

UNION ALL

SELECT 'fact_transactions', COUNT(*)
FROM fact_transactions
""").df()

display(warehouse_counts)

,table_name,row_count
0,dim_date,4018
1,dim_project,500
2,dim_employee,2231
3,dim_vendor,25
4,bridge_employee_project,42095
5,fact_transactions,50000


## Task 2.1 — Q1: Department Budget Performance

### Business Question

Which departments have spent more than 90% of their total allocated
budget, including departments that are already over budget?

### Required Output

- `department`
- `total_budget`
- `total_actual_cost`
- `spend_percentage`
- `over_budget`

### Approach

Use the `dim_project` warehouse dimension because it contains the project-level
budget and actual cost information.

1. Aggregate budget and actual cost by department.
2. Calculate spend percentage as:
   `total_actual_cost / total_budget * 100`
3. Keep departments where spend percentage is greater than 90%.
4. Flag departments as `over_budget = TRUE` when actual cost exceeds budget.
5. Order results by spend percentage in descending order.

The query is validated against the required column structure and the >90%
business rule.

In [36]:
# ============================================================
# Task 2.1 — Q1
# Department Budget Performance
# ============================================================

q1 = con.execute("""
    SELECT
        department,
        SUM(budget) AS total_budget,
        SUM(actual_cost) AS total_actual_cost,
        ROUND(
            SUM(actual_cost) * 100.0
            / NULLIF(SUM(budget), 0),
            2
        ) AS spend_percentage,
        CASE
            WHEN SUM(actual_cost) > SUM(budget)
            THEN TRUE
            ELSE FALSE
        END AS over_budget

    FROM dim_project

    GROUP BY department

    HAVING
        SUM(actual_cost) * 100.0
        / NULLIF(SUM(budget), 0) > 90

    ORDER BY spend_percentage DESC
""").df()

display(q1)

,department,total_budget,total_actual_cost,spend_percentage,over_budget
0,Legal,26490000.0,23959404.0,90.45,False


In [37]:
# ============================================================
# Q1 Validation
# ============================================================

expected_columns = [
    "department",
    "total_budget",
    "total_actual_cost",
    "spend_percentage",
    "over_budget"
]

# Validate required columns
assert list(q1.columns) == expected_columns, (
    f"Unexpected columns: {list(q1.columns)}"
)

# Validate business rule
assert (q1["spend_percentage"] > 90).all(), (
    "Q1 contains a department with spend percentage <= 90%"
)

# Validate over-budget flag
assert (
    q1["over_budget"]
    == (q1["total_actual_cost"] > q1["total_budget"])
).all(), "Incorrect over_budget flag detected"

# Validate ordering
assert q1["spend_percentage"].is_monotonic_decreasing, (
    "Results are not ordered by spend_percentage descending"
)

print("Q1 validation passed.")
print("Departments returned:", len(q1))

Q1 validation passed.
Departments returned: 1


### Q1 Result

The query successfully identifies departments whose actual spend exceeds
90% of their allocated budget.

The output satisfies the required column structure, >90% filtering rule,
over-budget flag logic, and descending spend-percentage ordering.

## Task 2.1 — Q2: Project Manager Workload

**Question:** Which managers currently oversee more than 3 active projects?

**Tables:** `dim_project`, `dim_employee`

**Approach:**
- Use `dim_employee` with `is_current = TRUE`.
- Join managers using `project_manager_id = employee_id`.
- Filter projects with `status = 'In Progress'`.
- Aggregate project count, budget and actual spend per manager.
- Return managers with more than 3 active projects.
- Order by `active_project_count` descending.

**Validation:** Check required columns, >3 project threshold, and ordering.

In [38]:
# ============================================================
# Task 2.1 — Q2
# Project Manager Workload
# ============================================================

q2 = con.execute("""
    SELECT
        e.full_name,
        e.email,
        COUNT(*) AS active_project_count,
        SUM(p.budget) AS combined_budget_responsibility,
        SUM(p.actual_cost) AS combined_actual_spend

    FROM dim_project p

    JOIN dim_employee e
        ON p.project_manager_id = e.employee_id
        AND e.is_current = TRUE

    WHERE p.status = 'In Progress'

    GROUP BY
        e.employee_id,
        e.full_name,
        e.email

    HAVING COUNT(*) > 3

    ORDER BY active_project_count DESC
""").df()

display(q2)

,full_name,email,active_project_count,combined_budget_responsibility,combined_actual_spend


In [39]:
# ============================================================
# Q2 Validation
# ============================================================

expected_columns = [
    "full_name",
    "email",
    "active_project_count",
    "combined_budget_responsibility",
    "combined_actual_spend"
]

# Validate required columns
assert list(q2.columns) == expected_columns, (
    f"Unexpected columns: {list(q2.columns)}"
)

# Validate business rule
assert (q2["active_project_count"] > 3).all(), (
    "Q2 contains a manager with 3 or fewer active projects"
)

# Validate ordering
assert q2["active_project_count"].is_monotonic_decreasing, (
    "Results are not ordered by active_project_count descending"
)

print("Q2 validation passed.")
print("Managers returned:", len(q2))

Q2 validation passed.
Managers returned: 0


In [40]:
# Q2 sanity check — active projects and manager assignment

q2_check = con.execute("""
    SELECT
        COUNT(*) AS active_projects,
        COUNT(project_manager_id) AS projects_with_manager,
        COUNT(DISTINCT project_manager_id) AS distinct_managers
    FROM dim_project
    WHERE status = 'In Progress'
""").df()

display(q2_check)

,active_projects,projects_with_manager,distinct_managers
0,189,189,148


In [41]:
q2_workload_check = con.execute("""
    SELECT
        p.project_manager_id,
        e.full_name,
        COUNT(*) AS active_project_count
    FROM dim_project p
    LEFT JOIN dim_employee e
        ON p.project_manager_id = e.employee_id
        AND e.is_current = TRUE
    WHERE p.status = 'In Progress'
    GROUP BY
        p.project_manager_id,
        e.full_name
    ORDER BY active_project_count DESC
    LIMIT 10
""").df()

display(q2_workload_check)

,project_manager_id,full_name,active_project_count
0,EMP0536,Nina Laurent,3
1,EMP0444,Hiroshi Malik,3
2,EMP0850,Divya Mehta,3
3,EMP0391,Aisha Mahmoud,3
4,EMP0192,Hassan Hamdan,3
5,EMP0092,Hassan Al Zaabi,3
6,EMP0852,Mateo Reddy,3
7,EMP0656,Hanan Al Rumaithi,3
8,EMP0513,Nina Rivera,3
9,EMP0109,Bilal Ibrahim,2


## Task 2.1 — Q3: Vendor Concentration Risk

**Question:** Which vendors account for more than 5% of total transaction spend?

**Tables:** `fact_transactions`, `dim_vendor`

**Approach:**
- Aggregate transaction spend by vendor.
- Calculate each vendor's percentage of total spend.
- Flag >10% as HIGH and 5–10% as MEDIUM.
- Return vendors above 5%, ordered by percentage descending.

**Validation:** Check required columns, >5% threshold, risk flag and ordering.

In [42]:
# ============================================================
# Task 2.1 — Q3
# Vendor Concentration Risk
# ============================================================

q3 = con.execute("""
    WITH vendor_spend AS (
        SELECT
            v.vendor_name,
            SUM(f.amount) AS total_spend,
            COUNT(*) AS transaction_count
        FROM fact_transactions f
        JOIN dim_vendor v
            ON f.vendor_key = v.vendor_key
        GROUP BY v.vendor_name
    ),

    total_spend AS (
        SELECT SUM(amount) AS total_amount
        FROM fact_transactions
    )

    SELECT
        vendor_name,
        total_spend,
        transaction_count,
        ROUND(
            total_spend * 100.0 / NULLIF(total_amount, 0),
            2
        ) AS percentage_of_total_spend,
        CASE
            WHEN total_spend * 100.0 / NULLIF(total_amount, 0) > 10
                THEN 'HIGH'
            WHEN total_spend * 100.0 / NULLIF(total_amount, 0) >= 5
                THEN 'MEDIUM'
            ELSE 'NORMAL'
        END AS risk_flag

    FROM vendor_spend
    CROSS JOIN total_spend

    WHERE total_spend * 100.0 / NULLIF(total_amount, 0) > 5

    ORDER BY percentage_of_total_spend DESC
""").df()

display(q3)

,vendor_name,total_spend,transaction_count,percentage_of_total_spend,risk_flag


In [43]:
# ============================================================
# Q3 Validation
# ============================================================

expected_columns = [
    "vendor_name",
    "total_spend",
    "transaction_count",
    "percentage_of_total_spend",
    "risk_flag"
]

assert list(q3.columns) == expected_columns

assert (q3["percentage_of_total_spend"] > 5).all()

assert (
    q3["risk_flag"]
    == q3["percentage_of_total_spend"].apply(
        lambda x: "HIGH" if x > 10
        else "MEDIUM" if x >= 5
        else "NORMAL"
    )
).all()

assert q3["percentage_of_total_spend"].is_monotonic_decreasing

print("Q3 validation passed.")
print("Vendors returned:", len(q3))

Q3 validation passed.
Vendors returned: 0


In [44]:
# Q3 sanity check — top vendors by spend

q3_check = con.execute("""
    SELECT
        v.vendor_name,
        SUM(f.amount) AS total_spend,
        COUNT(*) AS transaction_count,
        ROUND(
            SUM(f.amount) * 100.0
            / (SELECT SUM(amount) FROM fact_transactions),
            2
        ) AS percentage_of_total_spend
    FROM fact_transactions f
    JOIN dim_vendor v
        ON f.vendor_key = v.vendor_key
    GROUP BY v.vendor_name
    ORDER BY total_spend DESC
    LIMIT 10
""").df()

display(q3_check)

,vendor_name,total_spend,transaction_count,percentage_of_total_spend
0,Integra Tech,135120295.0,2024,4.36
1,QuantumByte,134912079.0,2004,4.36
2,PayGate MENA,133081114.0,2024,4.30
3,CX Dynamics,130823168.0,2015,4.23
4,TechBuild LLC,130022560.0,2012,4.20
5,AnalyticsPro,128117162.0,1975,4.14
6,Sage Advisory,126803425.0,2025,4.10
7,PowerScale,125870251.0,2009,4.07
8,DataSys Solutions,124963437.0,2011,4.04
9,MediaWorks,124717793.0,2027,4.03


## Task 2.1 — Q4: Projects with Open Financial Issues

**Question:** Which projects have pending or disputed transactions
totalling more than 50,000 AED?

**Tables:** `fact_transactions`, `dim_project`

**Approach:**
- Filter transactions with `payment_status` of `Pending` or `Disputed`.
- Aggregate transaction count and value by project.
- Join project details from `dim_project`.
- Keep projects with open transaction value > 50,000 AED.
- Order by open transaction value descending.

**Validation:** Check required columns, payment statuses, >50,000 AED threshold,
and ordering.

In [45]:
# ============================================================
# Task 2.1 — Q4
# Projects with Open Financial Issues
# ============================================================

q4 = con.execute("""
    SELECT
        p.project_id,
        p.project_name,
        p.department,
        p.status AS project_status,
        COUNT(*) AS open_transaction_count,
        SUM(f.amount) AS open_transaction_value

    FROM fact_transactions f

    JOIN dim_project p
        ON f.project_key = p.project_key

    WHERE f.payment_status IN ('Pending', 'Disputed')

    GROUP BY
        p.project_id,
        p.project_name,
        p.department,
        p.status

    HAVING SUM(f.amount) > 50000

    ORDER BY open_transaction_value DESC
""").df()

display(q4)

,project_id,project_name,department,project_status,open_transaction_count,open_transaction_value
0,PRJ0498,Visual Analytics Suite #498,Data Science,Completed,35,3886933.0
1,PRJ0164,Predictive Maintenance Phase 3 #164,Sales,Completed,29,3696478.0
2,PRJ0304,Customer Portal Redesign #304,Legal,Completed,31,3654834.0
3,PRJ0272,Fleet Management System Phase 2 #272,Data Science,Completed,25,3600482.0
4,PRJ0029,Citizen Services Portal Phase 3,Sustainability,Completed,28,3530929.0
...,...,...,...,...,...,...
430,PRJ0081,AI Forecasting Engine #81,IT,Completed,22,488862.0
431,PRJ0016,Energy Optimisation Platform - Q4,Engineering,In Progress,20,483062.0
432,PRJ0379,Data Lake Implementation #379,Sales,Completed,19,475539.0
433,PRJ0346,Contract Lifecycle Management Phase 3 #346,Marketing,Completed,18,395803.0


In [46]:
# ============================================================
# Q4 Validation
# ============================================================

expected_columns = [
    "project_id",
    "project_name",
    "department",
    "project_status",
    "open_transaction_count",
    "open_transaction_value"
]

assert list(q4.columns) == expected_columns

assert (q4["open_transaction_value"] > 50000).all()

assert q4["open_transaction_value"].is_monotonic_decreasing

print("Q4 validation passed.")
print("Projects returned:", len(q4))

Q4 validation passed.
Projects returned: 435


## Task 2.1 — Q5: Monthly Spend Trend

**Question:** What is the monthly transaction spend by category, with a
running total and month-over-month percentage change?

**Tables:** `fact_transactions`, `dim_date`

**Approach:**
- Aggregate transaction amount by month and category.
- Use a window `SUM()` for the running total within each category.
- Use `LAG()` to calculate month-over-month percentage change.
- Format the month as `YYYY-MM`.
- Order by category and month.

**Validation:** Check required columns, running totals and month ordering.

In [47]:
# ============================================================
# Task 2.1 — Q5
# Monthly Spend Trend with Running Total
# ============================================================

q5 = con.execute("""
    WITH monthly_spend AS (
        SELECT
            STRFTIME(d.full_date, '%Y-%m') AS year_month,
            f.category,
            SUM(f.amount) AS monthly_spend

        FROM fact_transactions f

        JOIN dim_date d
            ON f.date_key = d.date_key

        GROUP BY
            STRFTIME(d.full_date, '%Y-%m'),
            f.category
    ),

    calculated AS (
        SELECT
            year_month,
            category,
            monthly_spend,

            SUM(monthly_spend) OVER (
                PARTITION BY category
                ORDER BY year_month
                ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
            ) AS running_total,

            LAG(monthly_spend) OVER (
                PARTITION BY category
                ORDER BY year_month
            ) AS previous_month_spend

        FROM monthly_spend
    )

    SELECT
        year_month,
        category,
        monthly_spend,
        running_total,

        ROUND(
            (monthly_spend - previous_month_spend)
            * 100.0
            / NULLIF(previous_month_spend, 0),
            2
        ) AS month_over_month_pct_change

    FROM calculated

    ORDER BY
        category,
        year_month
""").df()

display(q5)

,year_month,category,monthly_spend,running_total,month_over_month_pct_change
0,2022-01,Cloud Services,793066.0,793066.0,NaN
1,2022-02,Cloud Services,292487.0,1085553.0,-63.12
2,2022-03,Cloud Services,573839.0,1659392.0,96.19
3,2022-04,Cloud Services,1149732.0,2809124.0,100.36
4,2022-05,Cloud Services,1609095.0,4418219.0,39.95
...,...,...,...,...,...
721,2026-01,Training,161789.0,197135427.0,-29.88
722,2026-02,Training,3283.0,197138710.0,-97.97
723,2026-03,Training,11038.0,197149748.0,236.22
724,2026-07,Training,14139.0,197163887.0,28.09


In [48]:
# ============================================================
# Q5 Validation
# ============================================================

expected_columns = [
    "year_month",
    "category",
    "monthly_spend",
    "running_total",
    "month_over_month_pct_change"
]

assert list(q5.columns) == expected_columns

# Validate YYYY-MM format
assert q5["year_month"].str.match(r"^\d{4}-\d{2}$").all()

# Running total should never decrease within a category
running_check = (
    q5.groupby("category")["running_total"]
    .apply(lambda x: x.is_monotonic_increasing)
)

assert running_check.all(), "Running total is not cumulative within a category"

print("Q5 validation passed.")
print("Rows returned:", len(q5))
print("Categories:", q5["category"].nunique())

Q5 validation passed.
Rows returned: 726
Categories: 14


## Task 2.1 — Q6: Employee Compensation History

**Question:** Which employees received the largest single salary increases?

**Table:** `dim_employee` (SCD Type 2)

**Approach:**
- Compare consecutive SCD2 employee records.
- Match the previous version to the new version using `employee_id`,
  `valid_to = valid_from`, and salary history.
- Calculate absolute and percentage salary increases.
- Return the top 20 increases by AED amount.

**Validation:** Check required columns, positive increases, and descending order.

In [51]:
# ============================================================
# Task 2.1 — Q6
# Employee Compensation History
# ============================================================

q6 = con.execute("""
    SELECT
        new.employee_id,
        new.full_name,
        new.valid_from AS change_date,
        old.salary AS previous_salary,
        new.salary AS new_salary,
        new.salary - old.salary AS increase_amount,
        ROUND(
            (new.salary - old.salary) * 100.0
            / NULLIF(old.salary, 0),
            2
        ) AS increase_pct

    FROM dim_employee new

    JOIN dim_employee old
        ON new.employee_id = old.employee_id
        AND old.valid_to = new.valid_from

    WHERE new.salary > old.salary

    ORDER BY increase_amount DESC

    LIMIT 20
""").df()

display(q6)

,employee_id,full_name,change_date,previous_salary,new_salary,increase_amount,increase_pct
0,EMP0059,Bilal Al Rumaithi,2009-02-13,33852.0,59572.0,25720.0,75.98
1,EMP0211,Mira Al Khaja,2014-05-28,45854.0,69078.0,23224.0,50.65
2,EMP0263,Wei Bhatt,2023-01-19,39557.0,62533.0,22976.0,58.08
3,EMP0599,Sami Al Mansoori,2018-07-27,42233.0,64777.0,22544.0,53.38
4,EMP0179,Khalid Al Naqbi,2009-11-01,40082.0,62270.0,22188.0,55.36
5,EMP0228,Nadia Al Suwaidi,2016-05-06,28836.0,49492.0,20656.0,71.63
6,EMP0936,Ahmed Mahmoud,2023-04-25,28742.0,48448.0,19706.0,68.56
7,EMP0890,Marcus Smith,2022-05-01,29591.0,49121.0,19530.0,66.00
8,EMP0921,Karthik Murthy,2023-04-12,28668.0,48034.0,19366.0,67.55
9,EMP0193,Dana Al Marri,2022-04-01,48120.0,67168.0,19048.0,39.58


In [52]:
# ============================================================
# Q6 Validation
# ============================================================

expected_columns = [
    "employee_id",
    "full_name",
    "change_date",
    "previous_salary",
    "new_salary",
    "increase_amount",
    "increase_pct"
]

assert list(q6.columns) == expected_columns
assert (q6["increase_amount"] > 0).all()
assert (q6["new_salary"] > q6["previous_salary"]).all()
assert len(q6) <= 20
assert q6["increase_amount"].is_monotonic_decreasing

print("Q6 validation passed.")
print("Salary increases returned:", len(q6))

Q6 validation passed.
Salary increases returned: 20


In [54]:
print(con.execute("SHOW TABLES").df())

                       name
0   bridge_employee_project
1                  dim_date
2              dim_employee
3               dim_project
4                dim_vendor
5              employees_df
6         fact_transactions
7               projects_df
8             stg_employees
9        stg_salary_history
10          transactions_df


In [57]:
print(con.execute("""
    SELECT column_name, data_type
    FROM information_schema.columns
    WHERE table_name = 'dim_project'
    ORDER BY ordinal_position
""").df())

               column_name      data_type
0              project_key        INTEGER
1               project_id        VARCHAR
2             project_name        VARCHAR
3               department        VARCHAR
4                   status        VARCHAR
5               start_date           DATE
6                 end_date           DATE
7                   budget  DECIMAL(15,2)
8              actual_cost  DECIMAL(15,2)
9       project_manager_id        VARCHAR
10                priority        VARCHAR
11                  region        VARCHAR
12         budget_variance  DECIMAL(15,2)
13          is_over_budget        BOOLEAN
14           duration_days        INTEGER
15  budget_utilisation_pct  DECIMAL(10,2)
16         status_category        VARCHAR
17              risk_level        VARCHAR


In [58]:
print(con.execute("""
    SELECT column_name, data_type
    FROM information_schema.columns
    WHERE table_name = 'fact_transactions'
    ORDER BY ordinal_position
""").df())

       column_name      data_type
0  transaction_key        INTEGER
1   transaction_id        VARCHAR
2      project_key        INTEGER
3     employee_key        INTEGER
4       vendor_key        INTEGER
5         date_key        INTEGER
6           amount  DECIMAL(15,2)
7         category        VARCHAR
8   payment_status        VARCHAR


In [61]:
# ============================================================
# Task 2.3 — 4a
# Original Query — EXPLAIN ANALYZE
# updated the tables names from the original query
# ============================================================

original_explain = con.execute("""
    EXPLAIN ANALYZE
    SELECT
        e.full_name,
        e.department,
        e.role,
        p.project_name,
        p.status,
        p.budget,
        p.actual_cost,
        t.amount,
        t.category,
        t.payment_status,
        d.full_date AS transaction_date

    FROM dim_employee e

    JOIN dim_project p
        ON e.employee_id = p.project_manager_id

    JOIN fact_transactions t
        ON p.project_key = t.project_key

    JOIN dim_date d
        ON t.date_key = d.date_key

    WHERE e.is_current = TRUE
      AND p.status NOT IN ('Completed', 'On Hold')
      AND t.payment_status = 'Pending'
      AND t.amount > (
            SELECT AVG(amount)
            FROM fact_transactions
            WHERE payment_status = 'Pending'
      )

    ORDER BY
        e.department,
        t.amount DESC
""").df()

print(original_explain.to_string(index=False))

plan_text = original_explain.iloc[0]["explain_value"]

evidence_path = (
    TASK2_OUTPUT_DIR / "task_2_3_original_explain.txt"
)

with open(evidence_path, "w", encoding="utf-8") as f:
    f.write(plan_text)

print(f"Saved to: {evidence_path}")


  explain_key                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           

In [62]:
# ============================================================================
# TASK 2.3 - OPTIMIZED QUERY + EXPLAIN ANALYZE
# ============================================================================

optimized_query = """
WITH pending_avg AS (
    SELECT AVG(amount) AS avg_pending_amount
    FROM fact_transactions
    WHERE payment_status = 'Pending'
)
SELECT
    e.full_name,
    e.department,
    e.role,
    p.project_name,
    p.status,
    p.budget,
    p.actual_cost,
    t.amount,
    t.category,
    t.payment_status,
    d.full_date AS transaction_date
FROM fact_transactions t
JOIN dim_project p
    ON p.project_key = t.project_key
JOIN dim_employee e
    ON e.employee_id = p.project_manager_id
    AND e.is_current = TRUE
JOIN dim_date d
    ON d.date_key = t.date_key
CROSS JOIN pending_avg a
WHERE p.status NOT IN ('Completed', 'On Hold')
  AND t.payment_status = 'Pending'
  AND t.amount > a.avg_pending_amount
ORDER BY
    e.department,
    t.amount DESC
"""

# --------------------------------------------------------------------------
# 1. Execute optimized query
# --------------------------------------------------------------------------

optimized_result = con.execute(
    optimized_query
).df()

print("Optimized query rows:", len(optimized_result))
print()
print("First 5 rows:")
print(optimized_result.head())


# --------------------------------------------------------------------------
# 2. Run EXPLAIN ANALYZE
# --------------------------------------------------------------------------

optimized_explain = con.execute(
    "EXPLAIN ANALYZE " + optimized_query
).df()

print()
print("=" * 80)
print("OPTIMIZED QUERY - EXPLAIN ANALYZE")
print("=" * 80)

print(optimized_explain.to_string(index=False))


# --------------------------------------------------------------------------
# 3. Save full execution plan as evidence
# --------------------------------------------------------------------------

optimized_plan_text = optimized_explain.iloc[0]["explain_value"]

optimized_evidence_path = (
    TASK2_OUTPUT_DIR / "task_2_3_optimized_explain.txt"
)

with open(
    optimized_evidence_path,
    "w",
    encoding="utf-8"
) as f:
    f.write(optimized_plan_text)

print()
print(f"Saved to: {optimized_evidence_path}")

Optimized query rows: 923

First 5 rows:
         full_name    department                 role  \
0      Mateo Reddy  Data Science         BI Developer   
1    Yusuf Mansour  Data Science          ML Engineer   
2      Roshan Iyer  Data Science          BI Engineer   
3  Hassan Al Zaabi  Data Science       Data Scientist   
4  Sami Al Bloushi  Data Science  Senior Data Analyst   

                              project_name       status     budget  \
0       Regulatory Compliance Tool Phase 3  In Progress   200000.0   
1  Legacy System Decommission Phase 3 #314  In Progress   800000.0   
2               Recommendation System - Q1  In Progress  2000000.0   
3   Sentiment Analysis Engine Phase 2 #101  In Progress  2000000.0   
4             Resource Scheduling Tool #62  In Progress   120000.0   

   actual_cost    amount    category payment_status transaction_date  
0     172385.0  795371.0    Security        Pending       2025-03-20  
1     676742.0  770822.0    Software        Pending  

In [ ]:
# ============================================================================
# TASK 2.3 - BEFORE / AFTER PERFORMANCE COMPARISON
# ============================================================================

# Extract execution times from EXPLAIN ANALYZE output
original_plan = original_explain.iloc[0]["explain_value"]
optimized_plan = optimized_explain.iloc[0]["explain_value"]

import re

original_time_match = re.search(
    r"Total Time:\s*([0-9.]+)s",
    original_plan
)

optimized_time_match = re.search(
    r"Total Time:\s*([0-9.]+)s",
    optimized_plan
)

original_time = float(original_time_match.group(1))
optimized_time = float(optimized_time_match.group(1))

improvement_pct = (
    (original_time - optimized_time)
    / original_time
) * 100

print("=" * 60)
print("TASK 2.3 - PERFORMANCE COMPARISON")
print("=" * 60)

print(f"Original execution time : {original_time:.4f} seconds")
print(f"Optimized execution time: {optimized_time:.4f} seconds")
print(f"Performance improvement : {improvement_pct:.2f}%")
print(f"Original rows           : {len(original_result)}")
print(f"Optimized rows          : {len(optimized_result)}")
print(
    f"Results match           : "
    f"{original_result.equals(optimized_result)}"
)

In [63]:
print(con.execute("SELECT version()").fetchone())

('v1.5.5',)


In [64]:
# ============================================================
# Task 2.3 — 4c
# Create production-recommended indexes
# ============================================================

indexes = [
    """
    CREATE INDEX IF NOT EXISTS idx_fact_transactions_payment_status
    ON fact_transactions (payment_status)
    """,
    """
    CREATE INDEX IF NOT EXISTS idx_fact_transactions_project_key
    ON fact_transactions (project_key)
    """,
    """
    CREATE INDEX IF NOT EXISTS idx_fact_transactions_date_key
    ON fact_transactions (date_key)
    """,
    """
    CREATE INDEX IF NOT EXISTS idx_dim_project_project_manager
    ON dim_project (project_manager_id)
    """,
    """
    CREATE INDEX IF NOT EXISTS idx_dim_project_status
    ON dim_project (status)
    """,
    """
    CREATE INDEX IF NOT EXISTS idx_dim_employee_employee_current
    ON dim_employee (employee_id, is_current)
    """
]

for index_sql in indexes:
    con.execute(index_sql)

print("All recommended indexes created successfully.")

All recommended indexes created successfully.


In [66]:
print(
    con.execute("""
        SELECT
            table_name,
            index_name,
            sql
        FROM duckdb_indexes()
        ORDER BY table_name, index_name
    """).df()
)

          table_name                            index_name  \
0       dim_employee     idx_dim_employee_employee_current   
1        dim_project       idx_dim_project_project_manager   
2        dim_project                idx_dim_project_status   
3  fact_transactions        idx_fact_transactions_date_key   
4  fact_transactions  idx_fact_transactions_payment_status   
5  fact_transactions     idx_fact_transactions_project_key   

                                                 sql  
0  CREATE INDEX idx_dim_employee_employee_current...  
1  CREATE INDEX idx_dim_project_project_manager O...  
2  CREATE INDEX idx_dim_project_status ON dim_pro...  
3  CREATE INDEX idx_fact_transactions_date_key ON...  
4  CREATE INDEX idx_fact_transactions_payment_sta...  
5  CREATE INDEX idx_fact_transactions_project_key...  


In [67]:
optimized_explain_indexed = con.execute(
    "EXPLAIN ANALYZE " + optimized_query
).df()

indexed_plan_text = optimized_explain_indexed.iloc[0]["explain_value"]

print(indexed_plan_text)

indexed_evidence_path = (
    TASK2_OUTPUT_DIR / "task_2_3_optimized_indexed_explain.txt"
)

with open(indexed_evidence_path, "w", encoding="utf-8") as f:
    f.write(indexed_plan_text)

print(f"Saved to: {indexed_evidence_path}")

┌─────────────────────────────────────┐
│┌───────────────────────────────────┐│
││    Query Profiling Information    ││
│└───────────────────────────────────┘│
└─────────────────────────────────────┘
EXPLAIN ANALYZE  WITH pending_avg AS (     SELECT AVG(amount) AS avg_pending_amount     FROM fact_transactions     WHERE payment_status = 'Pending' ) SELECT     e.full_name,     e.department,     e.role,     p.project_name,     p.status,     p.budget,     p.actual_cost,     t.amount,     t.category,     t.payment_status,     d.full_date AS transaction_date FROM fact_transactions t JOIN dim_project p     ON p.project_key = t.project_key JOIN dim_employee e     ON e.employee_id = p.project_manager_id     AND e.is_current = TRUE JOIN dim_date d     ON d.date_key = t.date_key CROSS JOIN pending_avg a WHERE p.status NOT IN ('Completed', 'On Hold')   AND t.payment_status = 'Pending'   AND t.amount > a.avg_pending_amount ORDER BY     e.department,     t.amount DESC 
┌─────────────────────────────

In [68]:
import time

# Original
start = time.perf_counter()
con.execute("""
SELECT
    e.full_name,
    e.department,
    e.role,
    p.project_name,
    p.status,
    p.budget,
    p.actual_cost,
    t.amount,
    t.category,
    t.payment_status,
    d.full_date AS transaction_date
FROM dim_employee e
JOIN dim_project p
    ON e.employee_id = p.project_manager_id
JOIN fact_transactions t
    ON p.project_key = t.project_key
JOIN dim_date d
    ON t.date_key = d.date_key
WHERE e.is_current = TRUE
  AND p.status NOT IN ('Completed', 'On Hold')
  AND t.payment_status = 'Pending'
  AND t.amount > (
        SELECT AVG(amount)
        FROM fact_transactions
        WHERE payment_status = 'Pending'
  )
ORDER BY e.department, t.amount DESC
""").df()

original_time = time.perf_counter() - start


# Optimized
start = time.perf_counter()
con.execute(optimized_query).df()

optimized_time = time.perf_counter() - start


print(f"Original runtime:  {original_time:.6f} seconds")
print(f"Optimized runtime: {optimized_time:.6f} seconds")

if optimized_time > 0:
    print(f"Speedup: {original_time / optimized_time:.2f}x")

Original runtime:  0.010095 seconds
Optimized runtime: 0.010809 seconds
Speedup: 0.93x
